In [ ]:
import numpy as np
import openpmd_viewer as ioview
import openpmd_api as io
%matplotlib widget
import scipy.constants as sc
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
ts = ioview.OpenPMDTimeSeries("./diags/fields/")

In [ ]:
ts.slider(cmap='RdBu_r',vmin=-1e6,vmax=1e6)

In [ ]:
770e6 * 11 / 1e9

In [ ]:
FE = pd.read_csv("./diags/reducedfiles/field_energy.txt", delimiter="\s+")

In [ ]:
FE

In [ ]:
FE.plot(x="[1]time(s)", y="[2]total_lev0(J)",logy=True)

In [ ]:
# The zero-th energy value is zero
time_s = FE["[1]time(s)"][1:]
energy_J = FE["[2]total_lev0(J)"][1:]

energy_dB = 10*np.log10(energy_J/np.max(energy_J))

In [ ]:
ns = 1e-9

fig,ax = plt.subplots(1,1)

ax.plot(time_s / ns, energy_dB)
ax.set_xlabel("Time (ns)")
ax.set_ylabel("EM energy in system (dB)")

In [ ]:
path   = f"./diags/receiving_plane/openpmd.bp5"
series = io.Series(path, )

In [ ]:
target = 2712   # the iteration you want

for it in series.read_iterations():
    if it.iteration_index < target:
        it.series_flush()   # must flush even if you don't load anything
        continue
    
    # now at the target iteration
    mesh = it.meshes["E"]
    comp = mesh["y"]
    raw  = comp.load_chunk()
    it.series_flush()
    print(f"Loaded iteration {it.iteration_index}, shape = {raw.shape}")
    break   # stop after the one you want

In [ ]:
gs   = mesh.grid_spacing    # [dx, dy, dz]
orig = mesh.grid_global_offset

In [ ]:
gs

In [ ]:
orig

In [ ]:
mesh.axis_labels

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from ipywidgets import interact, IntSlider
from openpmd_viewer import OpenPMDTimeSeries

ts = OpenPMDTimeSeries("./diags/fields/")

def plot_iteration(iteration_index):
    iteration = ts.iterations[iteration_index]
    
    # Load your field of interest
    Ex, info = ts.get_field(field='E', coord='z', slice_across='z', iteration=iteration)
    
    # Load eb_covered mask (1 = covered/metal, 0 = vacuum)
    eb, _ = ts.get_field(field='eb_covered', coord=None, slice_across='z', iteration=iteration)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Field plot
    im = ax.imshow(
        Ex,
        extent=info.imshow_extent,
        origin='lower',
        aspect=1.0,
        cmap='RdBu',
        vmin=-1e5,
        vmax=1e5,
    )
    plt.colorbar(im, ax=ax, label='Ez (V/m)')
    
    # eb_covered on top: 1=black (metal), 0=transparent (vacuum)
    # Build a colormap: 0 -> fully transparent, 1 -> black
    cmap_eb = mcolors.LinearSegmentedColormap.from_list(
        'eb_mask', [(0, 0, 0, 0), (0, 0, 0, 1)]  # (R,G,B,A) tuples
    )
    ax.imshow(
        eb,
        extent=info.imshow_extent,
        origin='lower',
        aspect=1.0,
        cmap=cmap_eb,
        vmin=0.0,
        vmax=1.0,
        interpolation='nearest',   # sharp edges on the geometry
    )
    
    ax.set_title(f'Iteration {iteration}')
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
    plt.tight_layout()
    plt.show()

interact(
    plot_iteration,
    iteration_index=IntSlider(min=0, max=len(ts.iterations)-1, step=1, value=0,
                              description='Iteration')
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import openpmd_api as io
from ipywidgets import interact, IntSlider

DIAG_PATH = "./diags/fields_sliced/openpmd.bp5/"
FIELD     = "E"
COORD     = "z"       # component: x, y, or z
SLICE_AX  = 0         # axis index to slice: 0=x, 1=y, 2=z
VMIN, VMAX = -1e5, 1e5

# ── 1. Read all iterations eagerly with read_linear ──────────────────────────
# We must consume the Series in one pass (variable-based encoding),
# so we cache every iteration's data into plain dicts up front.

iterations_idx = []   # iteration numbers in order
field_data     = {}   # iter -> 2D numpy array
eb_data        = {}   # iter -> 2D numpy array  (loaded from first iter only, then reused)
extents        = {}   # iter -> [xmin, xmax, ymin, ymax]

series = io.Series(DIAG_PATH, io.Access.read_linear)

eb_cache = None  # geometry is static — load once

for iteration in series.read_iterations():
    i = iteration.iteration_index
    iterations_idx.append(i)

    # ── Electric field component ─────────────────────────────────────────────
    mesh   = iteration.meshes[FIELD]
    comp   = mesh[COORD]

    # Geometry info: cell size and global offset
    geo    = mesh.geometry          # e.g. cartesian
    dx     = mesh.grid_spacing      # [dx, dy, dz]
    offset = mesh.grid_global_offset

    raw = comp.load_chunk()         # full 3-D array, shape [Nx, Ny, Nz]
    iteration.series_flush()

    # Convert to SI
    raw = raw * comp.unit_SI

    # Take central slice along SLICE_AX
    sl = [slice(None)] * raw.ndim
    sl[SLICE_AX] = raw.shape[SLICE_AX] // 2
    arr2d = raw[tuple(sl)]          # shape e.g. [Nx, Ny] when slicing z

    field_data[i] = arr2d

    # ── Extent in physical units ─────────────────────────────────────────────
    # Remaining axes after removing SLICE_AX
    axes = [a for a in range(raw.ndim) if a != SLICE_AX]
    ax0, ax1 = axes                 # e.g. 0 (x) and 1 (y)

    def axis_extent(ax, size):
        lo = offset[ax]
        hi = lo + dx[ax] * size
        return lo, hi

    x0, x1 = axis_extent(ax0, arr2d.shape[0])
    y0, y1 = axis_extent(ax1, arr2d.shape[1])
    extents[i] = [x0, x1, y0, y1]

    # ── eb_covered mask ──────────────────────────────────────────────────────
    if eb_cache is None and "eb_covered" in iteration.meshes:
        eb_mesh = iteration.meshes["eb_covered"]
        # eb_covered is a scalar field — component is the mesh itself
        eb_comp = eb_mesh[io.Mesh_Record_Component.SCALAR]
        eb_raw  = eb_comp.load_chunk()
        iteration.series_flush()
        sl_eb   = [slice(None)] * eb_raw.ndim
        sl_eb[SLICE_AX] = eb_raw.shape[SLICE_AX] // 2
        eb_cache = eb_raw[tuple(sl_eb)]

    eb_data[i] = eb_cache   # same geometry every iteration

del series   # close the file

# Fill any iterations where eb was missing (e.g. only written at t=0)
for i in iterations_idx:
    if eb_data.get(i) is None:
        eb_data[i] = eb_cache

# ── 2. Interactive slider ─────────────────────────────────────────────────────

cmap_eb = mcolors.LinearSegmentedColormap.from_list(
    'eb_mask', [(0, 0, 0, 0), (0, 0, 0, 1)]   # 0 → transparent, 1 → black
)

def plot_iteration(iteration_index):
    i   = iterations_idx[iteration_index]
    arr = field_data[i]
    eb  = eb_data[i]
    ext = extents[i]

    # imshow expects [xmin, xmax, ymin, ymax] but indexed as [row, col]
    # row = axis0 (x), col = axis1 (y) → extent = [y0, y1, x0, x1]
    imshow_extent = [ext[2], ext[3], ext[0], ext[1]]

    fig, ax = plt.subplots(figsize=(10, 6))

    im = ax.imshow(
        arr,
        extent=imshow_extent,
        origin='lower',
        aspect=1.0,
        cmap='RdBu',
        vmin=VMIN,
        vmax=VMAX,
    )
    plt.colorbar(im, ax=ax, label=f'E{COORD} (V/m)')

    if eb is not None:
        ax.imshow(
            eb,
            extent=imshow_extent,
            origin='lower',
            aspect=1.0,
            cmap=cmap_eb,
            vmin=0.0,
            vmax=1.0,
            interpolation='nearest',
        )

    ax.set_title(f'Iteration {i}')
    ax.set_xlabel('(m)')
    ax.set_ylabel('(m)')
    plt.tight_layout()
    plt.show()

interact(
    plot_iteration,
    iteration_index=IntSlider(
        min=0, max=len(iterations_idx) - 1, step=1, value=0,
        description='Iteration',
    ),
)

In [ ]:
s21_fft = np.load("./analysis/s21.npz",)

In [ ]:
s21_fft

In [ ]:
f0 = 67e9  # base frequency

In [ ]:
fig,ax = plt.subplots(1,1)

ax.plot(s21_fft["freqs"]/1e9,20*np.log10(np.abs(s21_fft["S21"])))
ax.axvline(x=f0/1e9, c="r", ls="--")

In [ ]:
data = np.load("./analysis/s21.npz", allow_pickle=True)

# --- FFT result (spectrum) ---
freqs = data["freqs"]
S21   = data["S21"]

# Wrapped phase: values stay in (-180, 180] — standard for S-params
phase_wrapped = np.degrees(np.angle(S21))

# Unwrapped phase: continuous, removes 2π jumps — useful to see
# total phase accumulation across frequency
phase_unwrapped = np.degrees(np.unwrap(np.angle(S21)))

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

ax = axes[0]
ax.plot(freqs * 1e-9, 20 * np.log10(np.abs(S21) + 1e-30))
ax.set_ylabel("|S21| (dB)")
ax.axvline(67, color="r", ls="--", label="67 GHz")
ax.legend(); ax.grid(True)

ax = axes[1]
ax.plot(freqs * 1e-9, phase_wrapped, label="wrapped")
ax.plot(freqs * 1e-9, phase_unwrapped, label="unwrapped", ls="--", c="C1")
ax.set_ylabel("∠S21 (deg)")
ax.set_xlabel("Frequency (GHz)")
ax.axhline(0, color="k", lw=0.5)
ax.axvline(67, color="r", ls="--")
ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig("s21_phase.png", dpi=150)
plt.show()

In [ ]:
data = np.load("./analysis/s21.npz", allow_pickle=True)

# --- FFT result (spectrum) ---
freqs = data["freqs"]
S21   = data["S21"]

# Wrapped phase: values stay in (-180, 180] — standard for S-params
phase_wrapped = np.degrees(np.angle(S21))

# Unwrapped phase: continuous, removes 2π jumps — useful to see
# total phase accumulation across frequency
phase_unwrapped = np.degrees(np.unwrap(np.angle(S21)))

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

for ax in axes:
    ax.set_xlim(60,70)

ax = axes[0]
ax.plot(freqs * 1e-9, 20 * np.log10(np.abs(S21) + 1e-30))
ax.set_ylabel("|S21| (dB)")
ax.axvline(67, color="r", ls="--", label="67 GHz")
ax.legend(); ax.grid(True)

ax = axes[1]
ax.plot(freqs * 1e-9, phase_wrapped, label="wrapped")
ax.set_ylabel("∠S21 (deg)")
ax.set_xlabel("Frequency (GHz)")
ax.axhline(0, color="k", lw=0.5)
ax.axvline(67, color="r", ls="--")
ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig("s21_phase_formatted.png", dpi=150)
plt.show()